<a href="https://colab.research.google.com/github/grasht/projects_ML_HW_4/blob/main/HW4_pt3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 3
**Part 1:**
I chose BERT embeddings for their ability to understand bidirectional context. This bidirectional learning of words seems like a powerful tool. Further, I do not see myself working with Fast Text because I do not have the need to work with morphologically rich languages. Word2Vec and Glove would have been fine choices too. The former provides nice simplicity while being a self supervised architecture that is quick to train. The latter provides global contextualization from its term-cooccurance matrices that also seem like a powerful ability, but ultimately BERT embeddings won out for providing a way to learn different meanings of the same word, something that I assume to be a very complicated, but important part of language.


**Sources:**

CodeEmporium - https://www.youtube.com/watch?v=EcuVZzUiHMY


Medium - https://medium.com/@davidlfliang/intro-getting-started-with-text-embeddings-using-bert-9f8c3b98dee6


In [ ]:
%pip install transformers torch


In [ ]:
from transformers import BertTokenizer, BertModel
import torch

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

text = input("Enter two words: ")
assert len(text.split()) == 2, "Input must be two words separated by white space."

inputs = tokenizer(text, return_tensors='pt')

with torch.no_grad():
  outputs = model(**inputs)

last_hidden_states = outputs.last_hidden_state

# Print embeddings for each token along with their vector dimension
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
print(f"\nTokens as generated by BERT: {tokens}")

embedding_dict = {}
prev_token = ''
for token, embedding in zip(tokens, last_hidden_states[0]):
  if token == '[CLS]' or token == '[SEP]':
    continue
  if token.startswith('##'):
    new_token = prev_token + token[2:]
    embedding_dict[new_token] = embedding_dict.pop(prev_token)
    embedding_dict[new_token].append(embedding.detach())
    prev_token = new_token

  else:
    prev_token = token
    embedding_dict[token] = [embedding.detach()]

print("\nFinal Embeddings: ")
for token, embedding in embedding_dict.items():
  if token == 'UNK':
    print("Warning: One of the given words is out of vocabulary. Try changing the spelling or form.")
  mean_embedding = torch.stack(embedding, dim=0).mean(dim=0)
  print(f"Token: {token}, Embedding Dimension: {mean_embedding.shape}.\nEmbedding: {mean_embedding}...")

**A note on OOV words:** Out of vocabulary (OOV) words are automatically broken down into sub tokens (when possible). Therefore, the mean of each subtoken embedding is taken above. If a word cannot be broken down like this, the UNK token is used and a warning is given.



# Part 2 - Cosine Similarity

Cosine similarity measures simularity by calculating the cosine of the angle between two non-zero vectors. A smaller distance indicates a higher similarity, while a larger one indicates the opposite. This is useful here because we can use this metric to compare pairs of words and reasonably estimate their similarity. This is a common metric for data analysis especially for document comparison, search queries, and recommendation systems.

Source: Geeks For Geeks - https://www.geeksforgeeks.org/dbms/cosine-similarity/

In [ ]:
from sklearn.manifold import TSNE
from matplotlib import pyplot as plt
import numpy as np

debug = False
text = input("Enter a list of word pairs, separated by commas: ")
pairs = [pair.strip() for pair in text.split(",")]

print(pairs)
valid = True
for pair in pairs:
    words = pair.split()  # split by whitespace
    if len(words) != 2:
        valid = False
        break

if not valid:
    raise ValueError("Each pair must contain exactly two words separated by a space.")

def cosine_similarity(vec1, vec2):
    return torch.dot(vec1, vec2) / (torch.norm(vec1) * torch.norm(vec2))

X = None
labels = []

for pair in pairs:
  inputs = tokenizer(pair, return_tensors='pt')

  with torch.no_grad():
    outputs = model(**inputs)

  last_hidden_states = outputs.last_hidden_state

  # Print embeddings for each token along with their vector dimension
  tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

  embedding_dict = {}
  prev_token = ''
  for token, embedding in zip(tokens, last_hidden_states[0]):
    if token == '[CLS]' or token == '[SEP]':
      continue

    if token.startswith('##'):
      new_token = prev_token + token[2:]
      embedding_dict[new_token] = embedding_dict.pop(prev_token)
      embedding_dict[new_token].append(embedding.detach())
      prev_token = new_token

    else:
      prev_token = token
      embedding_dict[token] = [embedding.detach()]

  final_embeddings = []
  words = []

  for token, embedding in embedding_dict.items():
    print(token)
    if token == 'UNK':
      print("Warning: One of the given words is out of vocabulary. Try changing the spelling or form.")
    mean_embedding = torch.stack(embedding, dim=0).mean(dim=0)
    final_embeddings.append(mean_embedding)
    words.append(token)
    if debug:
      print(mean_embedding)

  #Cosine Similarity
  similarity = cosine_similarity(final_embeddings[0], final_embeddings[1])
  print(f"Similarity between '{words[0]}' and '{words[1]}': {similarity}")

  X_curr = torch.stack(final_embeddings, dim=0).cpu().numpy()

  X = X_curr if X is None else np.concatenate((X, X_curr), axis=0)
  labels.extend(words)

#Convert to NumPy for t-SNE
tsne = TSNE(n_components=2, perplexity=1, random_state=7)
X_tsne = tsne.fit_transform(X)

#Plot
colors = [i - (i % 2) for i in range(len(labels))] #Color code the pairs
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=colors, cmap='viridis')
for i, token in enumerate(labels):
    plt.annotate(token, (X_tsne[i, 0], X_tsne[i, 1]))
plt.title("t-SNE of BERT Token Embeddings")
plt.xlabel("t-SNE Dim 1")
plt.ylabel("t-SNE Dim 2")
plt.show()

# Task 3

My solution to measuring dissimilarity was to try and average euclidean distance and cosine_similarity. With normalization they are pretty similar, so doing this has little impact on clustering. The code below allows you to change dist_type to work with either euclidean, cosine similarity, or the average, (1, 2, and 3 respectively).

I ran a test with two word pairings, for cat and the following three words: dog, lion, table. I recoreded the disimilarity measurements for each and plotted the results as bar graphs. As you can see they are rather similar.

A better implementation might be to measure dissimilarity between more than two words at a time. Then you can remove one word from the list and create a prototype martix. You can do this for each word and compare the smiliarty measurements with each word missing. The word with the lowest similarity may be an outlier. This however does not apply to two word embeddings.

Source:
[A Rank-Based Similarity Metric for Word Embeddings](https://aclanthology.org/P18-2088/) (Santus et al., ACL 2018)


In [ ]:
#Averaging Euclidean distance with Cosine Similarity

dist_type = 3

def euclidean_distance(vec1, vec2):
    return torch.sqrt(torch.sum((vec1 - vec2) ** 2))

def cosine_similarity(vec1, vec2):
    return torch.dot(vec1, vec2) / (torch.norm(vec1) * torch.norm(vec2))

def average_distance(vec1, vec2):
    return (euclidean_distance(vec1, vec2) + cosine_similarity(vec1, vec2)) / 2

def get_distance(vec1, vec2, dist_type = 1):
    if dist_type == 1:
        return euclidean_distance(vec1, vec2)
    elif dist_type == 2:
        return cosine_similarity(vec1, vec2)
    elif dist_type == 3:
        return average_distance(vec1, vec2)

debug = False
text = input("Enter a list of word pairs, separated by commas: ")
pairs = [pair.strip() for pair in text.split(",")]

print(pairs)
valid = True
for pair in pairs:
    words = pair.split()  # split by whitespace
    if len(words) != 2:
        valid = False
        break

if not valid:
    raise ValueError("Each pair must contain exactly two words separated by a space.")

X = None
labels = []

for pair in pairs:
  inputs = tokenizer(pair, return_tensors='pt')

  with torch.no_grad():
    outputs = model(**inputs)

  last_hidden_states = outputs.last_hidden_state

  # Print embeddings for each token along with their vector dimension
  tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

  embedding_dict = {}
  prev_token = ''
  for token, embedding in zip(tokens, last_hidden_states[0]):
    if token == '[CLS]' or token == '[SEP]':
      continue

    if token.startswith('##'):
      new_token = prev_token + token[2:]
      embedding_dict[new_token] = embedding_dict.pop(prev_token)
      embedding_dict[new_token].append(embedding.detach())
      prev_token = new_token

    else:
      prev_token = token
      embedding_dict[token] = [embedding.detach()]

  final_embeddings = []
  words = []

  for token, embedding in embedding_dict.items():
    print(token)
    if token == 'UNK':
      print("Warning: One of the given words is out of vocabulary. Try changing the spelling or form.")
    mean_embedding = torch.stack(embedding, dim=0).mean(dim=0)
    final_embeddings.append(mean_embedding)
    words.append(token)
    if debug:
      print(mean_embedding)

  #Cosine Similarity
  similarity = get_distance(final_embeddings[0], final_embeddings[1], dist_type)
  print(f"Similarity between '{words[0]}' and '{words[1]}': {similarity}")

  X_curr = torch.stack(final_embeddings, dim=0).cpu().numpy()

  X = X_curr if X is None else np.concatenate((X, X_curr), axis=0)
  labels.extend(words)

#Convert to NumPy for t-SNE
tsne = TSNE(n_components=2, perplexity=1, random_state=7)
X_tsne = tsne.fit_transform(X)

#Plot
colors = [i - (i % 2) for i in range(len(labels))] #Color code the pairs
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=colors, cmap='viridis')
for i, token in enumerate(labels):
    plt.annotate(token, (X_tsne[i, 0], X_tsne[i, 1]))
plt.title("t-SNE of BERT Token Embeddings")
plt.xlabel("t-SNE Dim 1")
plt.ylabel("t-SNE Dim 2")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

categories = ["Dog", "Lion", "Table"]
values = [12.76, 12.98, 13.06]

plt.bar(categories, values)

plt.xlabel("Word")
plt.ylabel("Value")
plt.title("Euclidean Distance from Cat")

plt.show()

categories = ["Dog", "Lion", "Table"]
values = [12.76, 12.98, 13.06]

plt.bar(categories, values)

plt.xlabel("Word")
plt.ylabel("Value")
plt.title("Cosine Similarity to Cat")

plt.show()

categories = ["Dog", "Lion", "Table"]
values = [6.66, 6.74, 6.76]

plt.bar(categories, values)

plt.xlabel("Word")
plt.ylabel("Value")
plt.title("Average Distance to Cat")

plt.show()